In [1]:
#Carichiamo QA, prodotti e recensioni del dataset Amazon-like
#Leggiamo solo le colonne utili per non appesantire troppo il notebook
#Stampiamo statistiche, colonne disponibili e copertura delle review testuali

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
INTERSECTION_DIR=PROJECT_ROOT/"data"/"processed"/"test_dataset2_intersection"
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/"test_dataset2_recommender"

ITEMS_PATH=INTERSECTION_DIR/"items.parquet"
QA_PATH=INTERSECTION_DIR/"qa.parquet"
RATINGS_PATH=INTERSECTION_DIR/"ratings.parquet"

items_df=pd.read_parquet(ITEMS_PATH)
qa_df=pd.read_parquet(QA_PATH)

ratings_columns=pq.ParquetFile(RATINGS_PATH).schema_arrow.names
wanted_rating_columns=["parent_asin","rating","review_title","review_text"]
available_rating_columns=[col for col in wanted_rating_columns if col in ratings_columns]
ratings_df=pd.read_parquet(RATINGS_PATH,columns=available_rating_columns)

print("ITEMS:",items_df.shape)
print("QA:",qa_df.shape)
print("RATINGS/REVIEWS:",ratings_df.shape)

print("\nColonne items:")
print(list(items_df.columns))

print("\nColonne qa:")
print(list(qa_df.columns))

print("\nColonne ratings/reviews lette:")
print(list(ratings_df.columns))

print("\nProdotti unici items:",items_df["parent_asin"].nunique() if "parent_asin" in items_df.columns else "colonna assente")
print("Prodotti unici QA:",qa_df["parent_asin"].nunique() if "parent_asin" in qa_df.columns else "colonna assente")
print("Prodotti unici reviews:",ratings_df["parent_asin"].nunique() if "parent_asin" in ratings_df.columns else "colonna assente")

if "review_text" in ratings_df.columns:
    non_empty_reviews=ratings_df["review_text"].dropna().astype(str).str.strip()
    non_empty_reviews=non_empty_reviews[non_empty_reviews!=""]
    print("\nReview testuali non vuote:",len(non_empty_reviews))
    print("Prodotti con almeno una review testuale:",ratings_df.loc[ratings_df["review_text"].notna(),"parent_asin"].nunique())
else:
    print("\nreview_text non presente")

ITEMS: (32266, 16)
QA: (256988, 12)
RATINGS/REVIEWS: (5245310, 4)

Colonne items:
['parent_asin', 'source_category', 'hf_category', 'qa_slug', 'category_group', 'title', 'category', 'avg_rating', 'num_ratings', 'price', 'store', 'features_json', 'description_json', 'categories_json', 'details_json', 'image_url']

Colonne qa:
['qa_id', 'parent_asin', 'source_category', 'hf_category', 'qa_slug', 'category_group', 'question', 'answer', 'questionType', 'answerType', 'answerTime', 'unixTime']

Colonne ratings/reviews lette:
['parent_asin', 'rating', 'review_title', 'review_text']

Prodotti unici items: 32266
Prodotti unici QA: 32266
Prodotti unici reviews: 32266

Review testuali non vuote: 5244869
Prodotti con almeno una review testuale: 32266


In [2]:
#Analizziamo parole e coppie di parole più frequenti
#Separiamo domande QA, risposte QA e recensioni
#Questa analisi serve a capire quali aspetti dei prodotti sono più presenti nei dati

from sklearn.feature_extraction.text import CountVectorizer

def find_col(df,candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def show_top_terms(name,series,ngram_range=(1,1),n=40,min_df=2):
    texts=series.dropna().astype(str).str.strip()
    texts=texts[texts!=""]
    if len(texts)==0:
        print(name,"| nessun testo disponibile")
        return pd.DataFrame()
    try:
        vectorizer=CountVectorizer(stop_words="english",ngram_range=ngram_range,min_df=min_df,max_features=50000)
        matrix=vectorizer.fit_transform(texts)
        counts=matrix.sum(axis=0).A1
        terms=vectorizer.get_feature_names_out()
        result_df=pd.DataFrame({"term":terms,"count":counts})
        result_df=result_df.sort_values("count",ascending=False).head(n)
        print("\n"+name)
        display(result_df)
        return result_df
    except ValueError as error:
        print(name,"| errore:",error)
        return pd.DataFrame()

question_col=find_col(qa_df,["question","question_text","q"])
answer_col=find_col(qa_df,["answer","answer_text","a"])
review_text_col=find_col(ratings_df,["review_text"])
review_title_col=find_col(ratings_df,["review_title"])

if question_col is not None:
    show_top_terms("Top parole nelle domande QA",qa_df[question_col],ngram_range=(1,1),n=40,min_df=2)
    show_top_terms("Top bigrammi nelle domande QA",qa_df[question_col],ngram_range=(2,2),n=40,min_df=2)

if answer_col is not None:
    show_top_terms("Top parole nelle risposte QA",qa_df[answer_col],ngram_range=(1,1),n=40,min_df=2)
    show_top_terms("Top bigrammi nelle risposte QA",qa_df[answer_col],ngram_range=(2,2),n=40,min_df=2)

if review_text_col is not None:
    show_top_terms("Top parole nelle recensioni",ratings_df[review_text_col],ngram_range=(1,1),n=40,min_df=5)
    show_top_terms("Top bigrammi nelle recensioni",ratings_df[review_text_col],ngram_range=(2,2),n=40,min_df=5)


Top parole nelle domande QA


,term,count
9473,does,70883
27251,work,42233
25970,use,18538
11310,fit,18274
18955,phone,15528
6783,case,12246
17503,need,11768
7509,come,11161
25425,tv,9425
6593,camera,9404



Top bigrammi nelle domande QA


,term,count
14714,does work,8708
13613,does come,5544
13807,does fit,3082
13986,does know,2127
14229,does phone,1857
39000,sim card,1854
49322,work windows,1721
37123,samsung galaxy,1720
13573,does case,1718
7873,case fit,1626



Top parole nelle risposte QA


,term,count
39231,yes,59997
36989,use,32821
38658,work,29011
20680,just,26931
13327,don,23289
13283,does,22950
26893,phone,19004
38686,works,18692
34129,sure,17827
24657,need,17771


... (2 outputs left out)


Top parole nelle recensioni


,term,count
8650,br,4740313
20904,great,1630251
20611,good,1180256
46851,use,1148787
25205,just,1075504
49041,works,976808
26505,like,958545
34657,product,841315
33074,phone,820652
49002,work,724400



Top bigrammi nelle recensioni


,term,count
4845,br br,1778340
49336,works great,266770
13909,easy use,114433
18492,great product,111571
40506,sound quality,96966
19822,highly recommend,87997
13801,easy install,78426
18480,great price,77753
3224,battery life,70979
38198,screen protector,66579


In [3]:
#Cerchiamo prodotti con molte QA e molte recensioni testuali
#Sono i prodotti migliori per provare il sistema QA
#La colonna copy_code contiene un parent_asin già pronto da copiare

qa_count=qa_df.groupby("parent_asin").size().rename("qa_count") if "parent_asin" in qa_df.columns else pd.Series(dtype=int)

if "parent_asin" in ratings_df.columns and "review_text" in ratings_df.columns:
    valid_reviews_df=ratings_df.dropna(subset=["review_text"]).copy()
    valid_reviews_df=valid_reviews_df[valid_reviews_df["review_text"].astype(str).str.strip()!=""]
    review_count=valid_reviews_df.groupby("parent_asin").size().rename("review_count")
else:
    review_count=pd.Series(dtype=int)

test_products_df=pd.concat([qa_count,review_count],axis=1).fillna(0).astype(int)
test_products_df["total_evidence"]=test_products_df["qa_count"]+test_products_df["review_count"]
test_products_df=test_products_df.sort_values(["total_evidence","qa_count","review_count"],ascending=False).head(30).reset_index()

if "title" in items_df.columns:
    titles_df=items_df[["parent_asin","title"]].drop_duplicates()
    test_products_df=test_products_df.merge(titles_df,on="parent_asin",how="left")

test_products_df["copy_code"]='PARENT_ASIN="'+test_products_df["parent_asin"].astype(str)+'"'

display(test_products_df)

,parent_asin,qa_count,review_count,total_evidence,title,copy_code
0,B00BWF5U0M,10,19364,19374,SABRENT 4 Port USB 2.0 Data Hub with Individua...,"PARENT_ASIN=""B00BWF5U0M"""
1,B0043T7FXE,10,15940,15950,Logitech M570 Wireless Trackball Mouse – Ergon...,"PARENT_ASIN=""B0043T7FXE"""
2,B005X1Y7I2,20,14748,14768,"Anker PowerCore+ Mini, 3350mAh Lipstick-Sized ...","PARENT_ASIN=""B005X1Y7I2"""
3,B00JRGOKQ8,10,14392,14402,iOttie Easy One Touch 2 Car Mount Holder Unive...,"PARENT_ASIN=""B00JRGOKQ8"""
4,B003NR57BY,10,12632,12642,Logitech M510 Wireless Computer Mouse – Comfor...,"PARENT_ASIN=""B003NR57BY"""
5,B00F0DD0I6,10,12555,12565,NETGEAR Nighthawk Smart Wi-Fi Router (R7000) -...,"PARENT_ASIN=""B00F0DD0I6"""
6,B00BIFNTMC,10,11319,11329,Anker 2.4G Wireless Vertical Ergonomic Optical...,"PARENT_ASIN=""B00BIFNTMC"""
7,B00IVPU786,10,11117,11127,Samsung 32GB up to 48MB/s EVO Class 10 Micro S...,"PARENT_ASIN=""B00IVPU786"""
8,B0092KJ9BU,10,11081,11091,Poly Voyager Legend Wireless Headset (Plantron...,"PARENT_ASIN=""B0092KJ9BU"""
9,B00IVPU7AO,10,11009,11019,Samsung EVO 64GB 48/MB/s Micro SDXC Memory Car...,"PARENT_ASIN=""B00IVPU7AO"""


In [4]:
#Mostriamo esempi reali di QA e recensioni
#Servono per capire la qualità del testo e il tipo di risposte disponibili
#Non usiamo user_id perché non serve per il QA

if question_col is not None and answer_col is not None:
    qa_sample_cols=["parent_asin",question_col,answer_col]
    display(qa_df[qa_sample_cols].dropna().sample(min(15,len(qa_df)),random_state=42))

if "review_text" in ratings_df.columns:
    review_sample_cols=[col for col in ["parent_asin","rating","review_title","review_text"] if col in ratings_df.columns]
    review_sample_df=ratings_df[review_sample_cols].dropna(subset=["review_text"])
    display(review_sample_df.sample(min(15,len(review_sample_df)),random_state=42))

,parent_asin,question,answer
239145,B002JE7FBW,Does this printer have an automatic double-sid...,The answer is yes. it also is fast and it work...
185788,B004ZL9IZI,lg500g phone: how do I switch from ringtone to...,The lower right hand key has a speaker symbol ...
5902,B0002YE2TE,what is the RMS wattage of this sub?,1000 watts rms full power 2000 watts
232730,B00007G7O1,Can this feed through multiple 6 part forms at...,I use this to print 1 six part form at a time ...
71207,B005110PXA,Will this work with the DVR player that record...,Don't no
16194,B000PALZE0,When are the prices for players coming DOWN???,I agree with you about Sony's products and mar...
75795,B005DKZTNK,will this works for windows 8?,"Yes. It works on all operating systems, or rea..."
28111,B001JTQCU8,Will It Work With a 1.0 USB hub? I need to bac...,My old USB 1.0 machines wouldn't recognize the...
30189,B001QWQDU2,Does it work on a dell inspiron 6000,"Maybe. If you are tech savvy, then it may work..."
30862,B001SN7QUU,vesa,Yes. Requires large size.


,parent_asin,rating,review_title,review_text
3158365,B000UIA6YM,5.0,Better by Far,This player has everything including a price t...
3227002,B004PNVETE,4.0,Good GPS,"I really liked this GPS, helps about finding t..."
145839,B007JCNO78,5.0,Ten out of five stars,Just installed this unit today. It's BT paire...
926983,B00BCSYZG4,1.0,"The blue light is nice, but...",The fan stopped working after 4 days of modera...
2889655,B006JH8T3S,5.0,Five Stars,easily best webcam I have ever bought.
3521213,B00D64RHMQ,5.0,Phone,Worked for a long time!
2108308,B0098PVH6E,5.0,Too Easy,Perfect.<br />I've purchased other scren prote...
974641,B008KYH9RA,3.0,Noisy,This pad is good in terms of cooling. It's sli...
2711263,B00I0G8GNY,1.0,A sad attempt to copy a lifeproof case,A sad attempt to copy a lifeproof case. This u...
4900290,B005IIR8HM,4.0,Four Stars,Good
